Installing peft, Uninstalling torchao

In [1]:
!pip install peft
!pip uninstall torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Would remove:
    /usr/local/lib/python3.13/dist-packages/torchao-0.10.0.dist-info/*
    /usr/local/lib/python3.13/dist-packages/torchao/*
Proceed (Y/n)? Y
  Successfully uninstalled torchao-0.10.0


Knowledge Distillation (Sequence) Step 2: Student Training

In [2]:
import json
import random
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

student_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
teacher_file = "/content/teacher_outputs.jsonl"
save_dir = "/content/distilled_model"

epochs = 3              # small dataset (140 kept examples), so several passes
lr = 2e-4
grad_accum = 8          # effective batch size 8
max_len = 1024
seed = 0

random.seed(seed)
torch.manual_seed(seed)

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Set the Colab runtime to T4.")

tokenizer = AutoTokenizer.from_pretrained(student_name)
model = AutoModelForCausalLM.from_pretrained(student_name, dtype=torch.float16, device_map="cuda")

prompt_template = ("Generate Google-style docstring for the following Python function: {code}. "
                   "Only output the docstring and function code.")   # P-2 wording

def prompt_ids(code):
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt_template.format(code=code)}],
                                         add_generation_prompt=True, tokenize=False)
    return tokenizer(text, add_special_tokens=False)["input_ids"]

# --- Coherence check on the test prompt (greedy), used before and after training
Code = '''from typing import List
def below_zero(operations: List[int]) -> bool:
    """ You're given a list of deposit and withdrawal operations
    on a bank account that starts with zero balance. Your task is to
    detect if at any point the balance of account falls below zero,
    and at that point function should return True.
    """
    balance = 0
    for op in operations:
        balance += op
        if balance < 0:
            return True
    return False'''

def coherence_check(label):
    model.eval()
    ids = torch.tensor([prompt_ids(Code)], device="cuda")
    with torch.no_grad():
        out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=300,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    n_new = out.shape[1] - ids.shape[1]
    print(f"\n--- Coherence check {label} ({n_new} tokens, limit 300) ---")
    print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

coherence_check("BEFORE distillation (plain TinyLlama)")

# --- Training data: the teacher's kept outputs, re-tokenized for the student's chat template
with open(teacher_file) as f:
    rows = [r for r in map(json.loads, f) if r["kept"]]
examples = []
for r in rows:
    p = prompt_ids(r["code"])
    a = tokenizer(r["output"].strip(), add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    if len(p) + len(a) <= max_len:
        examples.append((p + a, [-100] * len(p) + a))   # loss on the teacher's answer only
print(f"\nTraining examples: {len(examples)} (from {len(rows)} kept teacher outputs)")

# --- LoRA on all linear layers of the student's decoder blocks
# (a full fine-tune of 1.1B needs ~17 GB for weights, gradients and Adam states: too much for a T4)
model.config.use_cache = False
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
start = time.perf_counter()
step = 0
for epoch in range(epochs):
    model.train()
    random.shuffle(examples)
    running, count = 0.0, 0
    for i, (ids, labels) in enumerate(examples, 1):
        loss = model(input_ids=torch.tensor([ids], device="cuda"),
                     labels=torch.tensor([labels], device="cuda")).loss
        (loss / grad_accum).backward()
        running += loss.item()
        count += 1
        if i % grad_accum == 0 or i == len(examples):
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()
            optimizer.zero_grad()
            step += 1
    print(f"epoch {epoch + 1}/{epochs} | mean loss {running / count:.4f} | step {step}")

train_minutes = (time.perf_counter() - start) / 60
print(f"Training time: {train_minutes:.1f} min ({step} optimizer steps)")

# --- Merge the adapter into the FP16 weights (plain model at inference, no adapter overhead)
model = model.merge_and_unload()
model.config.use_cache = True
coherence_check("AFTER distillation")

model.save_pretrained(save_dir, max_shard_size="2GB")
print(f"\nDistilled student saved to {save_dir}")


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...


--- Coherence check BEFORE distillation (plain TinyLlama) (215 tokens, limit 300) ---
```python
def below_zero(operations: List[int]) -> bool:
    """
    You're given a list of deposit and withdrawal operations on a bank account that starts with zero balance. Your task is to
    detect if at any point the balance of account falls below zero, and at that point function should return True.
    """
    balance = 0
    for op in operations:
        balance += op
        if balance < 0:
            return True
    return False
```

This Google-style docstring for the function `below_zero` includes the following information:

- The function takes a list of deposit and withdrawal operations as input.
- The function checks if at any point the balance of the account falls below zero.
- The function returns a boolean value indicating whether the balance falls below zero or not.

The function code is also included, which is a simple loop that checks if the balance of the account falls below zer

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


epoch 3/3 | mean loss 0.0495 | step 54
Training time: 2.0 min (54 optimizer steps)

--- Coherence check AFTER distillation (223 tokens, limit 300) ---
Here is the Google-style docstring for the `below_zero` function:
```
def below_zero(operations: List[int]) -> bool:
    """
    You're given a list of deposit and withdrawal operations
    on a bank account that starts with zero balance. Your task is to
    detect if at any point the balance of account falls below zero,
    and at that point function should return True.

    Args:
        operations (List[int]): A list of deposit and withdrawal operations on a bank account that starts with zero balance.

    Returns:
        bool: True if the balance of the account falls below zero at any point, False otherwise.
    """
```
And here is the function code:
```
def below_zero(operations: List[int]) -> bool:
    balance = 0
    for op in operations:
        balance += op
        if balance < 0:
            return True
    return False
```


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]


Distilled student saved to /content/distilled_model
